# StatsBomb Extractor Test Suite

End-to-end test of every `extract_*` function in `libs/statsbomb/`.  
Each section calls the function, displays the resulting DataFrame, then builds and previews the JSON contract that `libs/footballd3/` consumes.

**Match:** UEFA Euro 2024 Final — Spain vs England  
**Run order matters:** `extract_xt` must run before `extract_momentum` (momentum reads `xt_actions_{match_id}.json`).

## Setup

In [ ]:
import sys, json
from pathlib import Path

project_root = Path().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root / "libs"))

import pandas as pd
from statsbombpy import sb

from statsbomb import (
    extract_shots,
    extract_freeze_frame,
    extract_convex_hull,
    extract_progressive_map,
    extract_possession,
    extract_heatmap,
    extract_pass_network,
    extract_formation,
    extract_team_shape_on_ball,
    extract_team_shape_off_ball,
    extract_xt,
    extract_momentum,
    extract_goal_animation,
    extract_play_animation,
    extract_match_stats,
)
from statsbomb.extract_heatmap import compute_kde_grid
from statsbomb.extract_xt import build_grid_json
from statsbomb.utils import resolve_match, fetch_match_info, build_nickname_lookup

SAMPLE_DATA = project_root / "libs" / "footballd3" / "sample_data"
SAMPLE_DATA.mkdir(parents=True, exist_ok=True)


import requests_cache

# Override statsbombpy's temp-dir cache with a persistent project-local cache.
# This survives kernel restarts; GitHub is only hit when data is stale (>24 h).
requests_cache.install_cache(
    str(project_root / ".cache" / "statsbombpy"),
    backend="sqlite",
    expire_after=86400,  # 24 hours
)

print(f"Project root: {project_root}")

## Inputs

In [ ]:
# Resolve the Euro 2024 Final — all functions below use this match_id.
MATCH_ID = resolve_match("UEFA Euro", "2024", "Spain", "England")

# Resolve home / away from the matches table.
comps = sb.competitions()
euro = comps[
    comps["competition_name"].str.contains("UEFA Euro", case=False)
    & (comps["season_name"] == "2024")
]
matches = sb.matches(
    competition_id=int(euro["competition_id"].iloc[0]),
    season_id=int(euro["season_id"].iloc[0]),
)
match_row = matches[matches["match_id"] == MATCH_ID].iloc[0]
HOME_TEAM = str(match_row["home_team"])
AWAY_TEAM = str(match_row["away_team"])

# Single-team inputs (Spain is used for progressive map, formation, pass network, team shape).
TEAM = "Spain"

# Possession to test extract_possession (picked for event-type diversity).
all_events = sb.events(match_id=MATCH_ID)
spain_poss = all_events[all_events["possession_team"] == TEAM]
POSSESSION_ID = int(
    spain_poss.groupby("possession")["type"]
    .nunique()
    .sort_values(ascending=False)
    .index[0]
)

print(f"MATCH_ID   : {MATCH_ID}")
print(f"HOME_TEAM  : {HOME_TEAM}")
print(f"AWAY_TEAM  : {AWAY_TEAM}")
print(f"TEAM       : {TEAM}")
print(f"POSSESSION_ID : {POSSESSION_ID}")

---
## 1. `extract_shots`

Extracts all Shot events. Returns one row per shot (own goals excluded — no xG).  
**JSON contract:** `shots_{match_id}.json` — array of `{x, y, xg, outcome, is_goal, team, display_name, minute}`.

In [ ]:
df_shots = extract_shots(MATCH_ID)
print(f"{len(df_shots)} shots — {df_shots['is_goal'].sum()} goals")
df_shots.head(10)

In [ ]:
# JSON contract — array of shot objects (event_id excluded from the contract).
shots_json = df_shots.drop(columns=["event_id"]).to_dict(orient="records")
out_path = SAMPLE_DATA / f"shots_{MATCH_ID}.json"
with open(out_path, "w") as f:
    json.dump(shots_json, f, indent=2)

print(f"Written: {out_path}")
print(json.dumps(shots_json[:2], indent=2))

---
## 2. `extract_freeze_frame`

Loads 360 frames for every goal event. Returns one row per visible player per goal.  
**JSON contract:** `freeze_frames_{match_id}_goals.json` — nested `{goals: [{ball, frame, visible_area, metadata}], match_metadata}`.

In [ ]:
from statsbomb.extract_freeze_frame import find_goal_frames, transform_frame
from statsbomb.utils import load_360_frames

df_ff = extract_freeze_frame(MATCH_ID)
print(f"{len(df_ff)} player-rows across {df_ff['goal_event_id'].nunique()} goal(s)")
df_ff.head(10)

In [ ]:
# JSON contract — reconstruct from raw ingredients (same as main()).
nicknames = build_nickname_lookup(MATCH_ID)
events = sb.events(match_id=MATCH_ID)
frame_lookup = load_360_frames(MATCH_ID)
competition, _, match_label = fetch_match_info(MATCH_ID)
goal_frames = find_goal_frames(events, frame_lookup)

snapshots = []
for row, frame in goal_frames:
    pid = row.get("player_id")
    display_name = nicknames.get(int(pid), str(row["player"])) if pid == pid else str(row["player"])
    snapshots.append(transform_frame(row, frame, MATCH_ID, display_name, competition, match_label))

ff_payload = {
    "goals": snapshots,
    "match_metadata": {"match_id": MATCH_ID, "competition": competition, "match_label": match_label},
}
out_path = SAMPLE_DATA / f"freeze_frames_{MATCH_ID}_goals.json"
with open(out_path, "w") as f:
    json.dump(ff_payload, f, indent=2)

print(f"Written: {out_path}  ({len(snapshots)} goal snapshots)")
# Preview: first goal — ball and first 3 players, then metadata.
g0 = snapshots[0]
preview = {**g0, "frame": g0["frame"][:3], "visible_area": f"<{len(g0['visible_area'])} coords>"}
print(json.dumps(preview, indent=2))

---
## 3. `extract_convex_hull`

Computes convex hulls for offense and defense at each goal instant (keepers excluded).  
**JSON contract:** `convex_hull_{match_id}_goals.json` — `{hulls: [{sides, metadata}], match_metadata}`.

In [ ]:
df_hull = extract_convex_hull(MATCH_ID)
print(f"{len(df_hull)} hull rows — {df_hull['goal_event_id'].nunique()} goal(s)")
# hull_vertices is a list; show its length so the df prints cleanly.
display_df = df_hull.copy()
display_df["hull_vertices"] = display_df["hull_vertices"].apply(lambda v: f"<{len(v)} pts>")
display_df

In [ ]:
# JSON contract — re-group by goal to reconstruct nested structure.
hull_entries = []
for event_id, group in df_hull.groupby("goal_event_id", sort=False):
    row0 = group.iloc[0]
    sides = [
        {"side": r["side"], "team_name": r["team_name"],
         "hull_vertices": r["hull_vertices"], "area": r["area"],
         "player_count": int(r["player_count"])}
        for _, r in group.iterrows()
    ]
    hull_entries.append({
        "sides": sides,
        "metadata": {
            "match_id": MATCH_ID, "event_id": str(event_id),
            "minute": int(row0["minute"]),
            "possession_team_name": str(row0["possession_team_name"]),
            "actor_team_name": str(row0["actor_team_name"]),
            "include_keeper": False,
        },
    })

hull_payload = {
    "hulls": hull_entries,
    "match_metadata": {"match_id": MATCH_ID, "competition": competition, "match_label": match_label},
}
out_path = SAMPLE_DATA / f"convex_hull_{MATCH_ID}_goals.json"
with open(out_path, "w") as f:
    json.dump(hull_payload, f, indent=2)

print(f"Written: {out_path}  ({len(hull_entries)} hull entries)")
# Preview: first hull — truncate vertices for readability.
h0 = hull_entries[0]
preview_sides = [{**s, "hull_vertices": s["hull_vertices"][:2] + ["..."]} for s in h0["sides"]]
print(json.dumps({**h0, "sides": preview_sides}, indent=2))

---
## 4. `extract_progressive_map`

All open-play passes and carries for one team. Each action has a `progressive` flag (25%-of-remaining-distance-to-goal rule).  
**JSON contract:** `progressive_map_{match_id}_{team_slug}.json` — `{team, actions, params, metadata}`.

In [ ]:
from statsbomb.extract_progressive_map import PROGRESSIVE_THRESHOLD

df_prog = extract_progressive_map(MATCH_ID, TEAM)
n_pass  = int((df_prog["action_type"] == "pass").sum())
n_carry = int((df_prog["action_type"] == "carry").sum())
n_prog  = int(df_prog["progressive"].sum())
print(f"{len(df_prog)} actions — {n_pass} passes, {n_carry} carries — {n_prog} progressive")
df_prog.head(10)

In [ ]:
import re

actions = df_prog.drop(columns=["event_id"]).to_dict(orient="records")
team_slug = re.sub(r"[^a-z0-9]+", "_", TEAM.lower()).strip("_")

prog_payload = {
    "team": TEAM,
    "actions": actions,
    "params": {"progressive_threshold": PROGRESSIVE_THRESHOLD},
    "metadata": {
        "match_id": MATCH_ID, "team": TEAM,
        "competition": competition, "match_label": match_label,
        "set_piece_filter": "play_pattern",
    },
}
out_path = SAMPLE_DATA / f"progressive_map_{MATCH_ID}_{team_slug}.json"
with open(out_path, "w") as f:
    json.dump(prog_payload, f, indent=2)

print(f"Written: {out_path}")
print(json.dumps({
    "team": prog_payload["team"],
    "actions": prog_payload["actions"][:2],
    "params": prog_payload["params"],
    "metadata": prog_payload["metadata"],
}, indent=2))

---
## 5. `extract_possession`

All events for a single possession. Consumed by `eventScatter.js` (spatial) and `timelineStrip.js` (temporal).  
**JSON contract:** `possession_{match_id}_{possession_id}.json` — `{match_id, possession, team, events, metadata}`.

In [ ]:
df_poss = extract_possession(MATCH_ID, POSSESSION_ID)
possession_team = df_poss.attrs["possession_team"]
print(f"Possession {POSSESSION_ID} — team: {possession_team} — {len(df_poss)} events — {df_poss['seconds'].max():.1f}s")
print(f"Event types: {sorted(df_poss['event_type'].unique())}")
df_poss

In [ ]:
poss_payload = {
    "match_id":   MATCH_ID,
    "possession": POSSESSION_ID,
    "team":       possession_team,
    "events":     df_poss.to_dict(orient="records"),
    "metadata": {
        "match_id": MATCH_ID, "possession": POSSESSION_ID,
        "team": possession_team, "competition": competition, "match_label": match_label,
    },
}
out_path = SAMPLE_DATA / f"possession_{MATCH_ID}_{POSSESSION_ID}.json"
with open(out_path, "w") as f:
    json.dump(poss_payload, f, indent=2)

print(f"Written: {out_path}")
print(json.dumps({
    **poss_payload,
    "events": poss_payload["events"][:3],
}, indent=2))

---
## 6. `extract_heatmap`

On-ball events for one player (auto-picks top event-count player). KDE density grid at 60×40 cells.  
**JSON contract:** `heatmap_{match_id}_{player_slug}.json` — `{grid: {cols, rows, values}, metadata}`.

In [ ]:
df_heat = extract_heatmap(MATCH_ID)  # auto-picks top-event-count player
display_name_heat = df_heat.attrs["display_name"]
team_heat = df_heat.attrs["team"]
print(f"Player: {display_name_heat} ({team_heat}) — {len(df_heat)} on-ball events")
print(f"Event types: {sorted(df_heat['event_type'].unique())}")
df_heat.head(10)

In [ ]:
bandwidth_yards, cols, rows = 5.0, 60, 40
events_for_kde = df_heat[["x", "y"]].to_dict(orient="records")
grid = compute_kde_grid(events_for_kde, bandwidth_yards=bandwidth_yards, cols=cols, rows=rows)

player_slug = re.sub(r"[^a-z0-9]+", "_", display_name_heat.lower()).strip("_")
heat_payload = {
    "grid": grid,
    "metadata": {
        "match_id": MATCH_ID, "display_name": display_name_heat, "team": team_heat,
        "competition": competition, "match_label": match_label,
        "event_count": len(df_heat), "method": "kde", "bandwidth_yards": bandwidth_yards,
        "grid_cols": cols, "grid_rows": rows,
        "pitch_width_yards": 120, "pitch_height_yards": 80,
    },
}
out_path = SAMPLE_DATA / f"heatmap_{MATCH_ID}_{player_slug}.json"
with open(out_path, "w") as f:
    json.dump(heat_payload, f, indent=2)

print(f"Written: {out_path}")
print(json.dumps({
    "grid": {"cols": grid["cols"], "rows": grid["rows"], "values": "<40×60 float grid>"},
    "metadata": heat_payload["metadata"],
}, indent=2))

---
## 7. `extract_pass_network`

Completed passes per ordered player pair, split by substitution windows. Returns one row per directed edge per window.  
**JSON contract:** `pass_network_{match_id}_{team}.json` — `{windows: [{index, label, nodes, edges}], substitutions, metadata}`.

In [ ]:
from statsbomb.extract_pass_network import _build_windows

df_pn = extract_pass_network(MATCH_ID, TEAM)
print(f"{len(df_pn)} edge rows across {df_pn['window_index'].nunique()} window(s)")
df_pn.head(10)

In [ ]:
windows, substitutions = _build_windows(all_events, TEAM, build_nickname_lookup(MATCH_ID))

pn_payload = {
    "windows": windows,
    "substitutions": substitutions,
    "metadata": {
        "match_id": MATCH_ID, "team": TEAM,
        "filter": "completed passes, per substitution window",
        "competition": competition, "match_label": match_label,
    },
}
out_path = SAMPLE_DATA / f"pass_network_{MATCH_ID}_{TEAM}.json"
with open(out_path, "w") as f:
    json.dump(pn_payload, f, indent=2)

print(f"Written: {out_path}")
w0 = windows[0]
print(json.dumps({
    "windows": [{
        **w0,
        "nodes": w0["nodes"][:3],
        "edges": w0["edges"][:3],
    }],
    "substitutions": substitutions[:2],
    "metadata": pn_payload["metadata"],
}, indent=2))

---
## 8. `extract_formation`

Ordered formation periods from Starting XI + Tactical Shift events. Template coordinates are canonical shape slots, not measured positions.  
**JSON contract:** `formation_{match_id}_{team_slug}.json` — `{periods: [{formation, from_minute, to_minute, players}], metadata}`.

In [ ]:
df_form = extract_formation(MATCH_ID, TEAM)
print(f"{len(df_form)} rows — formations: {df_form['formation'].unique().tolist()}")
df_form.head(15)

In [ ]:
from statsbomb.extract_formation import COORDINATE_NOTE

periods = []
for (formation, from_min, to_min), group in df_form.groupby(
    ["formation", "from_minute", "to_minute"], sort=False
):
    players = group.drop(columns=["formation", "from_minute", "to_minute"]).to_dict(orient="records")
    periods.append({
        "formation": formation, "from_minute": int(from_min),
        "to_minute": int(to_min), "players": players,
    })

form_payload = {
    "periods": periods,
    "metadata": {
        "match_id": MATCH_ID, "team": TEAM,
        "competition": competition, "match_label": match_label,
        "coordinate_note": COORDINATE_NOTE,
    },
}
team_slug = re.sub(r"[^a-z0-9]+", "_", TEAM.lower()).strip("_")
out_path = SAMPLE_DATA / f"formation_{MATCH_ID}_{team_slug}.json"
with open(out_path, "w") as f:
    json.dump(form_payload, f, indent=2)

print(f"Written: {out_path}  ({len(periods)} period(s))")
p0 = periods[0]
print(json.dumps({
    "periods": [{**p0, "players": p0["players"][:3]}],
    "metadata": form_payload["metadata"],
}, indent=2))

---
## 9. `extract_team_shape_on_ball` / `extract_team_shape_off_ball`

- **On-ball:** empirical mean positions from in-possession events, normalized so team attacks right.
- **Off-ball:** 360-frame dot density while out of possession (anonymous, camera-biased).
- **JSON contract:** `team_shape_{match_id}_{team_slug}.json` — `{on_ball, off_ball, metadata}`.

In [ ]:
df_on = extract_team_shape_on_ball(MATCH_ID, TEAM)
print(f"On-ball: {len(df_on)} rows — {df_on['period_from_minute'].nunique()} lineup period(s)")
df_on.head(15)

In [ ]:
off_ball = extract_team_shape_off_ball(MATCH_ID, TEAM)
print("Off-ball keys:", list(off_ball.keys()))
print(f"  centroid : {off_ball['centroid']}")
print(f"  thirds   : {off_ball['thirds_spine']}")
print(f"  ellipse  : {off_ball['ellipse']}")
print(f"  depth    : {off_ball['depth_line']}")
print(f"  grid     : {off_ball['density_grid']['rows']}×{off_ball['density_grid']['cols']} cells")

In [ ]:
periods_meta = df_on.attrs["periods_meta"]
on_ball_periods = []
for pm in periods_meta:
    group = df_on[
        (df_on["period_from_minute"] == pm["from_minute"])
        & (df_on["period_to_minute"] == pm["to_minute"])
    ]
    nodes = group.drop(columns=["period_from_minute", "period_to_minute"]).to_dict(orient="records")
    on_ball_periods.append({
        "from_minute": pm["from_minute"], "to_minute": pm["to_minute"],
        "players_in": pm["players_in"], "players_out": pm["players_out"],
        "nodes": nodes, "hull": pm["hull"],
    })

shape_payload = {
    "on_ball": {"periods": on_ball_periods},
    "off_ball": off_ball,
    "metadata": {
        "match_id": MATCH_ID, "team": TEAM,
        "competition": competition, "match_label": match_label,
        "on_ball_event_count": int(df_on["event_count"].sum()),
        "on_ball_period_count": len(on_ball_periods),
        "off_ball_bandwidth_yards": 8.0, "off_ball_depth_percentile": 70,
        "off_ball_grid_cols": 24, "off_ball_grid_rows": 16,
        "phase_filter": "open_play_only",
        "coordinate_system": "statsbomb_120x80_normalised_attack_right",
    },
}
team_slug = re.sub(r"[^a-z0-9]+", "_", TEAM.lower()).strip("_")
out_path = SAMPLE_DATA / f"team_shape_{MATCH_ID}_{team_slug}.json"
with open(out_path, "w") as f:
    json.dump(shape_payload, f, indent=2)

print(f"Written: {out_path}")
p0 = on_ball_periods[0]
print(json.dumps({
    "on_ball": {"periods": [{**p0, "nodes": p0["nodes"][:3], "hull": "<hull>"}]},
    "off_ball": {k: (v if k != "density_grid" else "<grid>") for k, v in off_ball.items()},
    "metadata": shape_payload["metadata"],
}, indent=2))

---
## 10. `extract_xt`

xT deltas for all open-play credited actions (completed passes + carries, both teams). Also emits the xT grid.  
**JSON contracts:** `xt_grid.json` and `xt_actions_{match_id}.json`.  
**Note:** `extract_momentum` reads `xt_actions_{match_id}.json` — run this cell before section 11.

In [ ]:
df_xt = extract_xt(MATCH_ID)
n_pass  = int((df_xt["action_type"] == "Pass").sum())
n_carry = int((df_xt["action_type"] == "Carry").sum())
n_pos   = int((df_xt["xt_delta"] > 0).sum())
print(f"{len(df_xt)} actions — {n_pass} passes, {n_carry} carries — {n_pos} positive xT")
df_xt.head(10)

In [ ]:
# Write xt_grid.json.
grid_data = build_grid_json()
grid_path = SAMPLE_DATA / "xt_grid.json"
with open(grid_path, "w") as f:
    json.dump(grid_data, f, indent=2)
print(f"Grid written: {grid_path}  ({grid_data['rows']}×{grid_data['cols']})")

# Write xt_actions_{match_id}.json.
xt_payload = {
    "actions": df_xt.to_dict(orient="records"),
    "metadata": {
        "match_id": MATCH_ID, "competition": competition,
        "match_label": match_label,
        "grid_source": "Karun Singh open_xt_12x8_v1",
        "grid_dims": [8, 12], "n_actions": len(df_xt),
    },
}
actions_path = SAMPLE_DATA / f"xt_actions_{MATCH_ID}.json"
with open(actions_path, "w") as f:
    json.dump(xt_payload, f, indent=2)
print(f"Actions written: {actions_path}")

# Preview one action record.
print(json.dumps(df_xt.head(1).to_dict(orient="records")[0], indent=2))

---
## 11. `extract_momentum`

Per-minute attacking momentum from the xT layer (reads `xt_actions_{match_id}.json` — run section 10 first).  
Smoothed with exponential decay window; includes goal and red-card annotations.  
**JSON contract:** `momentum_{match_id}.json` — `{home_team, away_team, minutes, secondary_minutes, goals, red_cards, params, metadata}`.

In [ ]:
df_mom = extract_momentum(MATCH_ID)
print(f"Home: {df_mom.attrs['home_team']}  Away: {df_mom.attrs['away_team']}")
print(f"{len(df_mom)} minutes  |  {len(df_mom.attrs['goals'])} goals  |  {len(df_mom.attrs['red_cards'])} red cards")
print(f"Params: {df_mom.attrs['params']}")
df_mom.head(10)

In [ ]:
mom_payload = {
    "home_team":         df_mom.attrs["home_team"],
    "away_team":         df_mom.attrs["away_team"],
    "minutes":           df_mom.to_dict(orient="records"),
    "secondary_minutes": df_mom.attrs["secondary_minutes"],
    "goals":             df_mom.attrs["goals"],
    "red_cards":         df_mom.attrs["red_cards"],
    "params":            df_mom.attrs["params"],
    "metadata": {
        "match_id": MATCH_ID, "competition": df_mom.attrs["competition"],
        "season": df_mom.attrs["season"], "match_label": df_mom.attrs["match_label"],
        "grid_source": df_mom.attrs["grid_source"],
    },
}
out_path = SAMPLE_DATA / f"momentum_{MATCH_ID}.json"
with open(out_path, "w") as f:
    json.dump(mom_payload, f, indent=2)

print(f"Written: {out_path}")
print(json.dumps({
    "home_team": mom_payload["home_team"],
    "away_team": mom_payload["away_team"],
    "minutes":   mom_payload["minutes"][:3],
    "goals":     mom_payload["goals"],
    "params":    mom_payload["params"],
    "metadata":  mom_payload["metadata"],
}, indent=2))

---
## 12. `extract_goal_animation`

Time-windowed ball-path clip (10 s before each goal, both teams' events). Ball paths are straight event-to-event segments — not real trajectories.  
**JSON contract:** `goal_animation_{match_id}.json` — `{goals: [{window, frames, context, metadata}], match_metadata}`.

In [ ]:
df_anim = extract_goal_animation(MATCH_ID)
n_goals = df_anim["goal_event_id"].nunique()
print(f"{n_goals} goal clip(s) — {len(df_anim)} frame rows total")
for gid, grp in df_anim.groupby("goal_event_id"):
    print(f"  Goal {grp.iloc[0]['goal_minute']}' — {grp.iloc[0]['goal_scorer']} ({grp.iloc[0]['goal_team']}) — {len(grp)} frames")
df_anim.head(10)

In [ ]:
clips = df_anim.attrs["clips"]
anim_payload = {
    "goals": clips,
    "match_metadata": {
        "match_id": MATCH_ID,
        "competition": df_anim.attrs["competition"],
        "match_label": df_anim.attrs["match_label"],
    },
}
out_path = SAMPLE_DATA / f"goal_animation_{MATCH_ID}.json"
with open(out_path, "w") as f:
    json.dump(anim_payload, f, indent=2)

print(f"Written: {out_path}")
c0 = clips[0]
print(json.dumps({
    "window":  c0["window"],
    "frames":  c0["frames"][:3],
    "context": c0["context"],
    "metadata": c0["metadata"],
}, indent=2))

---
## 13. `extract_play_animation`

General-purpose clip anchored to any event (not just goals). Uses the first goal's event_id as a demo anchor.  
**JSON contract:** same single-clip shape as one entry in `goals[]` above, with an empty `context`.

In [ ]:
# Use first goal event as the demo anchor.
first_goal_id = df_anim.loc[df_anim["goal_event_id"] == df_anim["goal_event_id"].iloc[0], "goal_event_id"].iloc[0]

df_play = extract_play_animation(MATCH_ID, first_goal_id, window_seconds=10.0)
print(f"Anchor: {first_goal_id}")
print(f"{len(df_play)} frame rows — t span: {df_play['t_seconds'].max():.2f}s")
df_play.head(10)

In [ ]:
clip = df_play.attrs["clip"]
print(json.dumps({
    "window":  clip["window"],
    "frames":  clip["frames"][:3],
    "context": clip["context"],
    "metadata": clip["metadata"],
}, indent=2))

---
## 14. `extract_match_stats`

Basic and advanced match statistics for both teams (shots, xG, possession, corners, cards, fouls).  
**JSON contract:** `match_stats_{match_id}.json` — `{home, away, rows, metadata}`.

In [ ]:
from statsbomb.extract_match_stats import load_team_colors, _DEFAULT_HOME_COLOR, _DEFAULT_AWAY_COLOR

df_stats = extract_match_stats(MATCH_ID)
home_team_s = df_stats.attrs["home_team"]
away_team_s = df_stats.attrs["away_team"]
print(f"{home_team_s} {df_stats.attrs['home_score']}–{df_stats.attrs['away_score']} {away_team_s}")
df_stats

In [ ]:
team_colors = load_team_colors(SAMPLE_DATA)

stats_payload = {
    "home": {
        "team":  home_team_s,
        "color": team_colors.get(home_team_s, _DEFAULT_HOME_COLOR),
        "score": df_stats.attrs["home_score"],
    },
    "away": {
        "team":  away_team_s,
        "color": team_colors.get(away_team_s, _DEFAULT_AWAY_COLOR),
        "score": df_stats.attrs["away_score"],
    },
    "rows": df_stats.to_dict(orient="records"),
    "metadata": {
        "match_id": MATCH_ID,
        "competition": df_stats.attrs["competition"],
        "match_label": df_stats.attrs["match_label"],
    },
}
out_path = SAMPLE_DATA / f"match_stats_{MATCH_ID}.json"
with open(out_path, "w") as f:
    json.dump(stats_payload, f, indent=2)

print(f"Written: {out_path}")
print(json.dumps(stats_payload, indent=2))

---
## Summary

| # | Function | DataFrame columns | JSON file |
|---|----------|-----------------|-----------|
| 1 | `extract_shots` | event_id, x, y, xg, outcome, is_goal, team, display_name, minute | `shots_{match_id}.json` |
| 2 | `extract_freeze_frame` | goal_event_id, goal_minute, scorer, scorer_team, ball_x/y, player_x/y, teammate, actor, keeper | `freeze_frames_{match_id}_goals.json` |
| 3 | `extract_convex_hull` | goal_event_id, minute, possession_team_name, actor_team_name, side, team_name, hull_vertices, area, player_count | `convex_hull_{match_id}_goals.json` |
| 4 | `extract_progressive_map` | event_id, action_type, display_name, x0/y0/x1/y1, completed, progressive, distance_gained, minute | `progressive_map_{match_id}_{team_slug}.json` |
| 5 | `extract_possession` | event_id, event_type, seconds, x, y, end_x, end_y, player, outcome | `possession_{match_id}_{possession_id}.json` |
| 6 | `extract_heatmap` | event_id, x, y, event_type, minute, display_name, team | `heatmap_{match_id}_{player_slug}.json` |
| 7 | `extract_pass_network` | window_index, window_label, from_player, from_x, from_y, to_player, count | `pass_network_{match_id}_{team}.json` |
| 8 | `extract_formation` | formation, from_minute, to_minute, player, display_name, jersey_number, position, template_x/y | `formation_{match_id}_{team_slug}.json` |
| 9a | `extract_team_shape_on_ball` | period_from/to_minute, player_id, player, display_name, x, y, event_count | `team_shape_{match_id}_{team_slug}.json` |
| 9b | `extract_team_shape_off_ball` | *(dict)* density_grid, centroid, thirds_spine, ellipse, depth_line | (same file) |
| 10 | `extract_xt` | event_id, team, display_name, minute, second, x0/y0/x1/y1, start_zone, end_zone, xt_delta, action_type | `xt_grid.json` + `xt_actions_{match_id}.json` |
| 11 | `extract_momentum` | minute, home_threat, away_threat, momentum | `momentum_{match_id}.json` |
| 12 | `extract_goal_animation` | goal_event_id, goal_minute, goal_scorer, goal_team, event_id, t_seconds, team, event_type, ball_x/y, ball_end_x/y, actor, outcome | `goal_animation_{match_id}.json` |
| 13 | `extract_play_animation` | (same as above, goal_* = None) | *(single clip dict via df.attrs["clip"])* |
| 14 | `extract_match_stats` | label, home_value, away_value, scale_type, format, tier | `match_stats_{match_id}.json` |